# Project Results

This notebook keeps the selected project-result workflow in one place: cluster-number diagnostics, algorithm comparison, Ward-method capacity results, and Ward-method time-series results.

It is intended for sharing the main results without carrying large NetCDF payloads or exploratory output cells.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "model_files").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SPORES_DIR = ROOT / "results" / "spores"
OUTPUT_DIR = ROOT / "outputs" / "project_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPORES_DIR, OUTPUT_DIR


In [ ]:
import pandas as pd

from calliope_nl_analysis.spores import list_spore_records, validate_spore_inventory
from calliope_nl_analysis.project_results import (
    CAPACITY_SETTINGS,
    TIMESERIES_SETTINGS,
    centroid_and_representative_statistics,
    elbow_curve,
    export_tables,
    fit_project_suite,
    hierarchical_linkages,
    nearest_neighbor_distances,
    parameter_table,
    pareto_candidates,
    pca_scores_and_loadings,
    percentile_selection,
    project_capacity_matrix,
    project_timeseries_matrix,
    validation_metrics,
    ward_result_tables,
)
from calliope_nl_analysis.plots import (
    plot_dendrograms,
    plot_elbow_curve,
    plot_capacity_centroids,
    plot_family_heatmap,
    plot_nearest_neighbor_curve,
    plot_pareto_3d,
    plot_pca_loadings,
    plot_pca_scores,
    plot_timeseries_centroids,
    plot_validation_metrics,
)


## Source Settings

These values document the project-result workflow explicitly, so the notebook remains reproducible and easy to review.


In [ ]:
settings_overview = pd.concat(
    {
        "capacity_flow_cap": parameter_table(CAPACITY_SETTINGS),
        "timeseries_cost_operation_variable": parameter_table(TIMESERIES_SETTINGS),
    },
    names=["dataset", "method"],
)
settings_overview

In [ ]:
inventory = validate_spore_inventory(SPORES_DIR)
records = list_spore_records(SPORES_DIR)
paths = [record.path for record in records]

inventory["total_files"], round(inventory["total_size_bytes"] / 1024**3, 2), inventory["missing_or_incomplete"]

## Capacity-Based Results

This workflow uses national-level `flow_cap` features for 7 source technologies and 2 storage technologies. Transmission, demand, losses, curtailment, imports, and exports are excluded from the project result tables.


In [ ]:
capacity = project_capacity_matrix(paths)
capacity.shape, capacity.head()


### Cluster Selection Diagnostics

These cells expose the cluster-number selection layer: K-means elbow, hierarchical dendrograms, and DBSCAN nearest-neighbor tuning.


In [ ]:
capacity_elbow = elbow_curve(capacity, CAPACITY_SETTINGS)
plot_elbow_curve(capacity_elbow, title="Capacity K-means elbow")

In [ ]:
capacity_linkages = hierarchical_linkages(capacity, CAPACITY_SETTINGS)
plot_dendrograms(
    capacity_linkages,
    cut_distances=CAPACITY_SETTINGS.hierarchical_cuts,
    title="Capacity hierarchical dendrograms",
)

In [ ]:
capacity_nn = nearest_neighbor_distances(capacity, CAPACITY_SETTINGS)
plot_nearest_neighbor_curve(
    capacity_nn,
    eps=CAPACITY_SETTINGS.dbscan_eps,
    title="Capacity DBSCAN nearest-neighbor distances",
)

In [ ]:
capacity_labels = fit_project_suite(capacity, CAPACITY_SETTINGS)
capacity_metrics = validation_metrics(
    capacity,
    capacity_labels,
    pca_components=CAPACITY_SETTINGS.pca_components,
).round(3)

capacity_params = parameter_table(CAPACITY_SETTINGS)
capacity_params = capacity_params.join(capacity_metrics[["clusters", "noise_points"]], rsuffix="_output")
capacity_params


In [ ]:
capacity_metrics

In [ ]:
plot_validation_metrics(capacity_metrics, title="Capacity clustering validation")

### Ward Interpretation

Ward's method is used for the capacity interpretation because it gives compact clusters while retaining the full SPORES allocation.


In [ ]:
capacity_results = ward_result_tables(capacity, CAPACITY_SETTINGS)
capacity_family_counts = capacity_results["family_counts"]
capacity_centroids = capacity_results["centroids"]
capacity_representatives = capacity_results["representatives"]

capacity_representatives

In [ ]:
plot_capacity_centroids(capacity_centroids, title="Capacity centroids, Ward method")

In [ ]:
plot_family_heatmap(capacity_family_counts, title="SPORES allocation in capacity clusters")

In [ ]:
capacity_scores_2d, capacity_loadings_2d, capacity_pca_2d, _ = pca_scores_and_loadings(capacity, n_components=2)
pd.Series(
    capacity_pca_2d.explained_variance_ratio_,
    index=["PC1", "PC2"],
    name="explained_variance_ratio",
).to_frame()

In [ ]:
plot_pca_scores(
    capacity_scores_2d,
    capacity_results["labelled_matrix"]["cluster"],
    title="Capacity PCA scores, Ward labels",
)

In [ ]:
capacity_pareto = pareto_candidates(
    capacity,
    capacity_results["labelled_matrix"]["cluster"],
    technologies=("ccgt", "coal", "bioenergy"),
    quantile=0.25,
)
capacity_pareto.head(20)

In [ ]:
plot_pareto_3d(capacity_pareto, technologies=("ccgt", "coal", "bioenergy"))

## Time-Series Operation-Cost Results

This workflow uses all yearly timesteps of `cost_operation_variable`, summed nationally and divided by the 3-hour timestep resolution to obtain average hourly operation costs.


In [ ]:
timeseries = project_timeseries_matrix(paths)
timeseries.shape, timeseries.iloc[:3, :5]


### Cluster Selection Diagnostics

The same selection diagnostics are repeated for the time-series matrix, where PCA reduces the yearly timestep features to two components.

In [ ]:
timeseries_elbow = elbow_curve(timeseries, TIMESERIES_SETTINGS)
plot_elbow_curve(timeseries_elbow, title="Time-series K-means elbow")

In [ ]:
timeseries_linkages = hierarchical_linkages(timeseries, TIMESERIES_SETTINGS)
plot_dendrograms(
    timeseries_linkages,
    cut_distances=TIMESERIES_SETTINGS.hierarchical_cuts,
    title="Time-series hierarchical dendrograms",
)

In [ ]:
timeseries_nn = nearest_neighbor_distances(timeseries, TIMESERIES_SETTINGS)
plot_nearest_neighbor_curve(
    timeseries_nn,
    eps=TIMESERIES_SETTINGS.dbscan_eps,
    title="Time-series DBSCAN nearest-neighbor distances",
)

In [ ]:
timeseries_labels = fit_project_suite(timeseries, TIMESERIES_SETTINGS)
timeseries_metrics = validation_metrics(
    timeseries,
    timeseries_labels,
    pca_components=TIMESERIES_SETTINGS.pca_components,
).round(3)

timeseries_params = parameter_table(TIMESERIES_SETTINGS)
timeseries_params = timeseries_params.join(timeseries_metrics[["clusters", "noise_points"]], rsuffix="_output")
timeseries_params


In [ ]:
timeseries_metrics

In [ ]:
plot_validation_metrics(timeseries_metrics, title="Time-series clustering validation")

### Ward Interpretation

Ward's method is used for the time-series interpretation because it gives a compact set of operation-cost patterns while retaining every SPORE.


In [ ]:
timeseries_results = ward_result_tables(timeseries, TIMESERIES_SETTINGS)
timeseries_family_counts = timeseries_results["family_counts"]
timeseries_centroids = timeseries_results["centroids"]
timeseries_representatives = timeseries_results["representatives"]

timeseries_representatives

In [ ]:
plot_timeseries_centroids(timeseries_centroids, title="Operation-cost centroids, Ward method")

In [ ]:
plot_family_heatmap(timeseries_family_counts, title="SPORES allocation in time-series clusters")

In [ ]:
timeseries_comparison = centroid_and_representative_statistics(
    timeseries,
    timeseries_results["labelled_matrix"]["cluster"],
    timeseries_representatives,
).round(2)
timeseries_comparison

## PCA-Based SPORES Selection

PCA scores and loadings expose the main operation-cost patterns. The cells below show both the component loadings and the percentile-based candidate selection used for decision-maker screening.


In [ ]:
scores, loadings, pca, scaler = pca_scores_and_loadings(timeseries, n_components=2)
explained_variance = pd.Series(
    pca.explained_variance_ratio_,
    index=["PC1", "PC2"],
    name="explained_variance_ratio",
)
explained_variance.to_frame()

In [ ]:
plot_pca_loadings(loadings, columns=("PC1", "PC2"), title="Time-series PCA loadings")

In [ ]:
plot_pca_scores(
    scores,
    timeseries_results["labelled_matrix"]["cluster"],
    title="Time-series PCA scores, Ward labels",
)

In [ ]:
pca_candidates = percentile_selection(scores, pc1_quantile=0.2, pc2_quantile=0.9)
pca_candidates.join(timeseries_results["labelled_matrix"][["cluster"]])

## Export Clean Tables

Run this cell to save only compact project-result tables. These are small CSV files, not the 23 GB NetCDF result payload.


In [ ]:
written = export_tables(
    {
        "capacity_parameters": capacity_params,
        "capacity_metrics": capacity_metrics,
        "capacity_elbow": capacity_elbow,
        "capacity_nearest_neighbor_distances": capacity_nn,
        "capacity_family_counts": capacity_family_counts,
        "capacity_centroids": capacity_centroids,
        "capacity_representatives": capacity_representatives,
        "capacity_low_dispatchable_candidates": capacity_pareto,
        "timeseries_parameters": timeseries_params,
        "timeseries_metrics": timeseries_metrics,
        "timeseries_elbow": timeseries_elbow,
        "timeseries_nearest_neighbor_distances": timeseries_nn,
        "timeseries_family_counts": timeseries_family_counts,
        "timeseries_representative_comparison": timeseries_comparison,
        "timeseries_pca_explained_variance": explained_variance,
        "timeseries_pca_candidates": pca_candidates,
    },
    OUTPUT_DIR,
)
written